# MMLU baseline for the original model

This notebook measures MMLU performance of the loaded model **before any editing**. It calls `evaluate_model_on_mmlu(...)` directly, so it does not reuse the cached baseline from the experiment pipeline.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

from heretic.config import Settings
from heretic.model import Model
from evaluate.mmlu import evaluate_model_on_mmlu

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# Change these values when you want to compare modes.
MMLU_CONFIG = {
    "enabled": True,
    "dataset": "cais/mmlu",
    "subset": "all",
    "split": "test",
    "mode": "zero_shot",
    "answer_mode": "logits",   # "generate" or "logits"
    "n_shots": 0,
    "sample_size": 100,
    "sample_seed": 42,
    "max_new_tokens": 32,
    "store_predictions": True,
}

original_argv = sys.argv.copy()
try:
    sys.argv = [sys.argv[0]] if sys.argv else ["notebook"]
    settings = Settings(
        model=MODEL_NAME,
        batch_size=16,
        max_response_length=2048,
        system_prompt="You are a helpful assistant.",
    )
finally:
    sys.argv = original_argv

model = Model(settings)

# Direct call: evaluates the currently loaded model as-is, without any editing.
result = evaluate_model_on_mmlu(model, MMLU_CONFIG)
summary = result["summary"]

print("MMLU summary for the original model")
print(json.dumps(summary, indent=2, ensure_ascii=False))

predictions = result.get("predictions", result.get("prediction_preview", []))
preview_rows = []
for row in predictions[:20]:
    preview_rows.append(
        {
            "subject": row.get("subject"),
            "question": row.get("question"),
            "correct_letter": row.get("correct_letter"),
            "predicted_letter": row.get("predicted_letter"),
            "is_correct": row.get("is_correct"),
            "raw_response": row.get("raw_response"),
            "choice_scores": row.get("choice_scores"),
        }
    )

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

output_dir = Path("results") / "notebooks"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "mmlu_original_model_baseline.json"
output_file.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved full baseline result to: {output_file}")
